# Лабораторная работа №1: Настройка AI-окружения и первый вызов API
**Дисциплина:** Искусственный интеллект  
**Студент:** Мыльников Александр Русланович, группа ФИТ-221  
**Тема диплома:** Разработка системы неразрушающего контроля для выявления дефектов металлических бутылок на конвейерной линии

In [3]:
!pip install --upgrade pip requests python-dotenv

  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\user\Desktop\univer_8\ml\ai-course-labs\.venv\Scripts\python.exe -m pip install --upgrade pip requests python-dotenv


In [5]:
import os
import sys
import logging
from typing import Dict
from datetime import datetime
import requests
from dotenv import load_dotenv

# Настройка логирования
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Загрузка переменных из .env
load_dotenv()
print('✅ Переменные окружения загружены')

✅ Переменные окружения загружены


In [6]:
class YandexGPTClient:
    """Клиент для взаимодействия с YandexGPT API."""
    
    def __init__(self, iam_token: str, folder_id: str):
        if not iam_token or not folder_id:
            raise ValueError("Необходимо указать iam_token и folder_id")
        self.iam_token = iam_token
        self.folder_id = folder_id
        self.model_uri = f"gpt://{folder_id}/yandexgpt/latest"
        self.api_url = "https://llm.api.cloud.yandex.net/foundationModels/v1/completion"
        logger.info(f"Клиент инициализирован. Folder ID: {folder_id}")

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 1000) -> Dict:
        """Генерация ответа модели."""
        headers = {
            "Content-Type": "application/json",
            "Authorization": f"Bearer {self.iam_token}",
            "x-folder-id": self.folder_id
        }
        payload = {
            "modelUri": self.model_uri,
            "completionOptions": {
                "stream": False,
                "temperature": temperature,
                "maxTokens": max_tokens
            },
            "messages": [
                {
                    "role": "system",
                    "text": "Вы — полезный ассистент. Отвечайте точно и по делу. Используйте русский язык."
                },
                {
                    "role": "user",
                    "text": prompt
                }
            ]
        }

        logger.info(f"Отправка запроса. Длина промпта: {len(prompt)} символов")
        try:
            response = requests.post(self.api_url, headers=headers, json=payload, timeout=30)
            response.raise_for_status()
            result = response.json()
            
            if "result" not in result:
                raise ValueError("Некорректный формат ответа API")
            alternatives = result["result"].get("alternatives", [])
            if not alternatives:
                raise ValueError("Пустой ответ от модели")
            
            generated_text = alternatives[0]["message"]["text"]
            tokens_info = result["result"].get("usage", {})
            
            response_data = {
                "text": generated_text,
                "tokens_input": tokens_info.get("inputTextTokens", 0),
                "tokens_output": tokens_info.get("completionTokens", 0),
                "raw_response": result
            }
            logger.info(f"Запрос выполнен. Выходных токенов: {response_data['tokens_output']}")
            return response_data
        except requests.exceptions.Timeout:
            logger.error("Превышено время ожидания ответа от API")
            raise
        except requests.exceptions.RequestException as e:
            logger.error(f"Ошибка запроса: {e}")
            raise
        except ValueError as e:
            logger.error(f"Ошибка парсинга ответа: {e}")
            raise

    def test_connection(self) -> bool:
        """Проверка подключения к API."""
        try:
            test_prompt = "Ответь одним словом: работает"
            response = self.generate(test_prompt, temperature=0.1)
            return "работает" in response["text"].lower()
        except Exception as e:
            logger.error(f"Тест подключения не пройден: {e}")
            return False

In [7]:
# Инициализация клиента
IAM_TOKEN = os.getenv("YANDEX_IAM_TOKEN")
FOLDER_ID = os.getenv("YANDEX_FOLDER_ID")

if not IAM_TOKEN or not FOLDER_ID:
    print("❌ ОШИБКА: Укажите YANDEX_IAM_TOKEN и YANDEX_FOLDER_ID в файле .env")
    sys.exit(1)

client = YandexGPTClient(IAM_TOKEN, FOLDER_ID)
print("✅ Клиент инициализирован")

2026-05-05 18:56:54,477 - INFO - Клиент инициализирован. Folder ID: b1ghag5n93mrclmq57u7


✅ Клиент инициализирован


In [8]:
# Проверка подключения
print("🔄 Проверка подключения...")
if client.test_connection():
    print("✅ Подключение успешно")
else:
    print("❌ Подключение не удалось")
    sys.exit(1)

2026-05-05 18:56:58,677 - INFO - Отправка запроса. Длина промпта: 29 символов


🔄 Проверка подключения...


2026-05-05 18:56:59,612 - INFO - Запрос выполнен. Выходных токенов: 2


✅ Подключение успешно


### Тестовый запрос
Базовый вызов API с общим вопросом об искусственном интеллекте.

In [9]:
test_prompt = "Объясни кратко, что такое искусственный интеллект (не более 100 слов)"
print(f"Запрос: {test_prompt}\n")

response = client.generate(test_prompt, temperature=0.5)

print("ОТВЕТ МОДЕЛИ:")
print("-" * 80)
print(response["text"])
print("-" * 80)
print(f"Статистика:")
print(f"  • Входные токены: {response['tokens_input']}")
print(f"  • Выходные токены: {response['tokens_output']}")
print(f"  • Время: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

2026-05-05 18:57:04,782 - INFO - Отправка запроса. Длина промпта: 69 символов


Запрос: Объясни кратко, что такое искусственный интеллект (не более 100 слов)



2026-05-05 18:57:06,852 - INFO - Запрос выполнен. Выходных токенов: 91


ОТВЕТ МОДЕЛИ:
--------------------------------------------------------------------------------
Искусственный интеллект (ИИ) — это область компьютерных наук, которая занимается разработкой интеллектуальных компьютерных систем, способных выполнять задачи, требующие человеческого интеллекта. К таким задачам относятся обучение, планирование, решение проблем, понимание естественного языка и машинное зрение. ИИ может быть основан на различных алгоритмах и методах, включая нейронные сети, машинное обучение и экспертные системы. Основная цель искусственного интеллекта — создание машин, способных имитировать человеческое поведение и мышление для решения разнообразных задач.
--------------------------------------------------------------------------------
Статистика:
  • Входные токены: 47
  • Выходные токены: 91
  • Время: 2026-05-05 18:57:06


### Адаптированный запрос по теме диплома

In [10]:
def get_specialty_prompt() -> str:
    """Промпт, адаптированный под специальность."""
    prompt = """
    Я студент технической специальности, работаю над дипломной темой:
    "Разработка системы неразрушающего контроля для выявления дефектов (брака) металлических бутылок на конвейерной линии"

    Прошу предоставить информацию по следующим вопросам:
    1. Какие виды критических дефектов (например, трещины сварного шва, коррозия, раковины, нарушение геометрии) характерны для металлических бутылок в зависимости от способа их изготовления (штамповка, сварка, вытяжка), и какие из них могут быть надежно выявлены методами неразрушающего контроля в движении на конвейере?
    2. Какой метод неразрушающего контроля (вихретоковый, магнитопорошковый, акустический, оптический с предобработкой изображений для металлических бликующих поверхностей) является оптимальным для непрерывной инспекции металлических бутылок на конвейере с точки зрения скорости (режим реального времени), чувствительности к микродефектам и устойчивости к вибрациям линии?
    3. Как разработать алгоритм автоматической классификации «годен/брак» на основе сигналов от датчиков (или изображений) металлических бутылок, чтобы минимизировать вероятность ложной отбраковки (false positive) при высокой скорости движения конвейера, учитывая естественный разброс параметров (например, толщина металла или магнитные свойства) для разных партий тары?

    Требования к ответу:
    • Ответ должен быть структурирован
    • Используй технические термины
    • Приведи конкретные примеры
    • Объём: 500-1000 слов
    """
    return prompt

# Выполнение адаптированного запроса
specialty_prompt = get_specialty_prompt()
print("ЗАПРОС:")
print("-" * 80)
print(specialty_prompt)
print("-" * 80)

spec_response = client.generate(specialty_prompt, temperature=0.5)

print("\nОТВЕТ МОДЕЛИ:")
print("-" * 80)
print(spec_response["text"])
print("-" * 80)
print(f"Токены: вход={spec_response['tokens_input']}, выход={spec_response['tokens_output']}")

# Сохранение результата
with open("specialty_response.txt", "w", encoding="utf-8") as f:
    f.write("ЗАПРОС:\n" + specialty_prompt + "\n\nОТВЕТ:\n" + spec_response["text"])
print("\n✅ Результат сохранён в specialty_response.txt")

2026-05-05 18:57:11,522 - INFO - Отправка запроса. Длина промпта: 1483 символов


ЗАПРОС:
--------------------------------------------------------------------------------

    Я студент технической специальности, работаю над дипломной темой:
    "Разработка системы неразрушающего контроля для выявления дефектов (брака) металлических бутылок на конвейерной линии"

    Прошу предоставить информацию по следующим вопросам:
    1. Какие виды критических дефектов (например, трещины сварного шва, коррозия, раковины, нарушение геометрии) характерны для металлических бутылок в зависимости от способа их изготовления (штамповка, сварка, вытяжка), и какие из них могут быть надежно выявлены методами неразрушающего контроля в движении на конвейере?
    2. Какой метод неразрушающего контроля (вихретоковый, магнитопорошковый, акустический, оптический с предобработкой изображений для металлических бликующих поверхностей) является оптимальным для непрерывной инспекции металлических бутылок на конвейере с точки зрения скорости (режим реального времени), чувствительности к микродефекта

2026-05-05 18:57:23,031 - INFO - Запрос выполнен. Выходных токенов: 796



ОТВЕТ МОДЕЛИ:
--------------------------------------------------------------------------------
### 1. Виды критических дефектов металлических бутылок и методы их выявления

**Дефекты при различных способах изготовления:**

- **Штамповка:**
  - Нарушение геометрии: деформация, несоответствие размеров.
  - Трещины: микротрещины, возникающие из-за напряжений в материале.
  - Раковины: пустоты или углубления в металле.

- **Сварка:**
  - Трещины сварного шва: неполное проплавление, поры, свищи.
  - Коррозия: очаги коррозии в зоне сварного шва.
  - Нарушение геометрии: несоответствие размеров и формы сварного соединения.

- **Вытяжка:**
  - Нарушение геометрии: неравномерная толщина стенок, искажение формы.
  - Трещины: микротрещины из-за напряжений в материале.
  - Дефекты поверхности: царапины, вмятины.

**Методы неразрушающего контроля для выявления дефектов:**

- **Вихретоковый метод:** эффективен для обнаружения трещин, раковин и нарушения геометрии в металлических изделиях. Подходит 